# ReleaseBot in Google Colab
**CoreSmart.AI · 17-Week GenAI Developer Program · Week 1**

This notebook runs the **exact same ReleaseBot code** as the local lab, just inside Colab instead of on your own machine. Nothing about the app changes: same FastAPI service, same `/summarize` and `/summarize-stream` endpoints, same browser UI.

> **Which path should I use?** If you can, run it on your **own machine** (that is the path the video walks through, and it is the setup you will use for the graded project). Use this Colab notebook when you cannot install Python locally, or you just want to try it fast in the browser.

You will do four things: **add your secrets**, **get the code**, **start the server**, then **exercise the API**. Run the cells top to bottom.

---
## 1 · Add your secrets (do this first)

ReleaseBot needs your own OpenAI key and, to actually send the summary email, your own Gmail credentials. **Never paste secrets into a code cell.** Colab has a built-in secrets vault instead.

Click the **key icon** (Secrets) in the left sidebar, add the three secrets below, and turn **Notebook access** on for each:

| Secret name | What it is |
|---|---|
| `OPENAI_API_KEY` | Your OpenAI API key (starts `sk-`). |
| `SMTP_SENDER` | The Gmail address the summary is sent from. |
| `SMTP_PASSWORD` | A Gmail **App Password** (16 chars). Needs 2-Step Verification on. Create one at myaccount.google.com/apppasswords. |

> The server needs all three set to boot. The Gmail ones are only actually used when you call `/summarize` (the email step). If you just want to try streaming, you can put a placeholder in the two SMTP secrets for now.

The next cell reads those secrets into the environment. If a secret is missing it will quietly prompt you for it (the input is hidden). It prints only whether each value was found, never the value itself.

In [ ]:
import os, getpass

def _load(name, prompt):
    val = None
    try:
        from google.colab import userdata  # present only inside Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None  # secret not added, or notebook access is off
    except ImportError:
        pass  # not running in Colab
    if not val:
        val = getpass.getpass(prompt)  # hidden input, not echoed
    return val

os.environ['OPENAI_API_KEY'] = _load('OPENAI_API_KEY', 'OpenAI API key: ')
os.environ['SMTP_SENDER']   = _load('SMTP_SENDER',   'Gmail sender address: ')
os.environ['SMTP_PASSWORD'] = _load('SMTP_PASSWORD', 'Gmail app password: ')

# Confirm without revealing anything.
for k in ['OPENAI_API_KEY', 'SMTP_SENDER', 'SMTP_PASSWORD']:
    v = os.environ.get(k, '')
    print(f'{k:15s}: ' + ('set (' + str(len(v)) + ' chars)' if v else 'MISSING'))

---
## 2 · Get the code and install dependencies

This clones the course labs repo and moves into the Week 1 ReleaseBot folder. If the clone fails (for example the repo is private and you are not signed in), use the upload fallback described just below the cell.

In [ ]:
import os

REPO_URL = 'https://github.com/coresmartai/labs.git'
SUBDIR   = 'week-01/releasebot'

if not os.path.isdir('labs'):
    !git clone --depth 1 $REPO_URL labs

target = os.path.join('labs', SUBDIR)
if os.path.isdir(target):
    os.chdir(target)
print('Working directory:', os.getcwd())
print('App present     :', os.path.isfile('app/main.py'))

> **Clone did not work?** Download this `releasebot` folder as a zip, then run a cell with:
> ```python
> from google.colab import files; files.upload()   # pick your zip
> !unzip -q releasebot.zip && cd releasebot
> ```
> and re-run the check above so `app/main.py` is found.

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

### (optional) Run the tests
A quick confidence check that the app is wired correctly before you start it. Four tests should pass.

In [ ]:
!python -m pytest -q

---
## 3 · Start the server

On your own machine you would run `uvicorn app.main:app --reload` in a terminal. Colab has no separate terminal, so this cell starts the same server **in the background** and waits until its `/health` check answers.

In [ ]:
import subprocess, time, sys, urllib.request, json

# Start uvicorn in the background, logging to a file we can tail on failure.
log = open('uvicorn.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '127.0.0.1', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT,
)

# Poll /health for up to ~30s.
url = 'http://127.0.0.1:8000/health'
ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            print('Server is up:', json.loads(r.read().decode()))
            ready = True
            break
    except Exception:
        time.sleep(1)

if not ready:
    print('Server did not start. Last log lines:')
    print(open('uvicorn.log').read()[-1500:])

### Open the browser UI (optional)
This exposes the running server through **Colab's own port proxy**, scoped to your session. Click the link it prints to use the same UI shown in the video.

> **Heads up on streaming in this view:** the proxy can buffer Server-Sent Events, so the browser UI may show the streamed summary arrive in one chunk rather than token by token. The token-by-token behaviour is real; the streaming cell further down proves it by talking to the server directly. The one-round-trip `/summarize` flow works fully in the UI.

In [ ]:
try:
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8000)
except ImportError:
    print('Not in Colab. Open http://127.0.0.1:8000 in your browser.')

---
## 4 · Exercise the API from here

These cells call the server **directly at `localhost`**, not through the proxy, so streaming arrives token by token exactly as designed. This is the same set of calls as the local `week1_notebook.ipynb`.

In [ ]:
import requests, json, textwrap

BASE      = 'http://127.0.0.1:8000'
RECIPIENT = 'training@coresmart.ai'  # change to your own email to receive it

DEMO_NOTES = textwrap.dedent('''
    v2.4 - Fixed login retry loop on Safari that locked users out after 3 failed attempts.
    Adjusted session token TTL from 1h to 4h to reduce re-authentication friction.
    Added experimental dark mode toggle in user settings (opt-in only).
    Patched XSS vulnerability in the comment renderer - update strongly recommended.
    Bumped Node.js runtime to 20 LTS.
''').strip()
print('Ready. Notes length:', len(DEMO_NOTES), 'chars')

### 4.1 · Health check `GET /health`

In [ ]:
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

### 4.2 · Streaming summary `POST /summarize-stream`
Pure token streaming over SSE, no tool call. Watch it arrive frame by frame.

In [ ]:
payload = {'release_notes': DEMO_NOTES}
print('-- Streaming output --\n')
with requests.post(f'{BASE}/summarize-stream', json=payload, stream=True) as resp:
    resp.raise_for_status()
    for raw in resp.iter_lines():
        if not raw:
            continue
        line = raw.decode() if isinstance(raw, bytes) else raw
        if not line.startswith('data: '):
            continue
        body = line[6:]
        if body == '[DONE]':
            print('\n\n-- Stream complete --')
            break
        try:
            print(json.loads(body).get('delta', ''), end='', flush=True)
        except json.JSONDecodeError:
            pass

### 4.3 · Structured summary and tool call `POST /summarize`
One round trip: the forced `send_email` tool call's arguments **are** the structured summary, and the email actually goes out (if your Gmail secrets are real).

In [ ]:
payload = {'release_notes': DEMO_NOTES, 'recipient': RECIPIENT}
print('Calling POST /summarize ...')
r = requests.post(f'{BASE}/summarize', json=payload)
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    s = data['summary']
    print('\n-- Summary --')
    print('Headline   :', s['headline'])
    print('Risk level :', s['risk_level'])
    print('Bullets    :')
    for b in s['bullets']:
        print('  -', b)
    print('\n-- Tool calls --')
    for tc in data['tool_calls']:
        print('  ', tc['name'], '->', tc['result'])

### 4.4 · Failure mode: empty notes
Pydantic enforces `min_length=1`, so the server returns **422** before the model is ever called. No tokens spent on a bad request.

In [ ]:
r = requests.post(f'{BASE}/summarize', json={'release_notes': '', 'recipient': RECIPIENT})
print('Status:', r.status_code, ' (expected 422)')
print(json.dumps(r.json(), indent=2))

---
### When you are done
Stop the background server to free the runtime:

In [ ]:
server.terminate()
print('Server stopped.')